# 04 Evaluate before freeze

Class coverage under both candidates, threshold class cross-tab against RRK (evaluation only), covariate balance sample versus frame, representativeness, practicality counts by owner and access class, and the QA report. Shengli's model test happens outside this notebook; its result and the one design iteration are recorded in `docs/DESIGN_SUMMARY.md`.

In [ ]:
import sys, os
print(sys.executable)
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
from src.io import load_config, get_logger
from src import strata, qa
cfg = load_config()
log = get_logger("04_evaluate")
log.info(f"Project: {cfg['project']['name']} | synthetic={cfg['run']['synthetic']} | freeze={cfg['run']['freeze']}")
P = Path(cfg["paths"]["processed"]); O = Path(cfg["paths"]["outputs"]); P.mkdir(parents=True, exist_ok=True); O.mkdir(exist_ok=True)

In [ ]:
frame = pd.read_parquet(P / "frame_classified.parquet")
sel = pd.read_parquet(P / "selected.parquet")
alloc = pd.read_csv(O / "allocation_table.csv")
prim = sel[sel["status"].isin(["primary", "legacy"])]

## 1. Class coverage by level

In [ ]:
cov = prim.groupby(["level", "cell_id"]).size().unstack("level", fill_value=0)
cov["cum_min"] = cov.get("min", 0); cov["cum_option"] = cov["cum_min"] + cov.get("option", 0); cov["cum_full"] = cov["cum_option"] + cov.get("full", 0)
floor = cfg["allocation"]["floor_per_cell"]
cov["below_floor_min"] = cov["cum_min"] < floor["min"]
log.info(f"Cells below floor at min: {int(cov['below_floor_min'].sum())} of {len(cov)}")
cov.to_csv(O / "eval_class_coverage.csv"); cov

## 2. Threshold class check (RRK, evaluation only)

Every VP9 and VP10 class, attaining and non-attaining, must be represented per forest type at the minimum quantity. On the synthetic run the RRK classes are stubbed from the proxies.

In [ ]:
if "rrk_seral" not in prim:
    prim = prim.assign(rrk_seral=prim["seral_class"], rrk_cover=prim["cover_class"],
                       rrk_density_attain=np.where(prim["density_class"] == "high", "exceeds", "meets"))
xt1 = pd.crosstab([prim["forest_type"], prim["rrk_seral"]], prim["rrk_cover"])
xt2 = pd.crosstab([prim["forest_type"], prim["rrk_seral"]], prim["rrk_density_attain"])
missing = int((xt2 == 0).sum().sum())
log.info(f"Empty forest type x seral x density-attainment combinations: {missing}")
xt1.to_csv(O / "eval_vp9_crosstab.csv"); xt2.to_csv(O / "eval_vp10_crosstab.csv"); xt2

## 3. Covariate balance: sample vs frame (KS test and standardized mean difference)

In [ ]:
from scipy import stats
covs = [c for c in ["elev_m", "slope_pct", "aspect_deg", "p95_height_m", "canopy_cover_pct", "stem_density", "dist_road_m"] if c in frame]
rows = []
for c in covs:
    for ftype in ["ALL"] + sorted(frame["forest_type"].unique()):
        f = frame if ftype == "ALL" else frame[frame["forest_type"] == ftype]
        s = prim if ftype == "ALL" else prim[prim["forest_type"] == ftype]
        if len(s) < 5: continue
        ks = stats.ks_2samp(s[c].dropna(), f[c].dropna())
        smd = (s[c].mean() - f[c].mean()) / f[c].std()
        rows.append({"covariate": c, "forest_type": ftype, "n_sample": len(s), "ks_stat": round(ks.statistic, 3), "ks_p": round(ks.pvalue, 3), "smd": round(smd, 3)})
bal = pd.DataFrame(rows)
flag = bal[(bal["smd"].abs() > 0.25) & (bal["ks_p"] < 0.05)]
log.info(f"Covariate imbalance flags (|SMD|>0.25 and KS p<0.05): {len(flag)}")
bal.to_csv(O / "eval_covariate_balance.csv", index=False); flag

## 4. Representativeness (when Shengli's raster is available)

In [ ]:
if "representativeness" in frame:
    q = np.quantile(frame["representativeness"], [0.25, 0.5, 0.75]); qs = np.quantile(prim["representativeness"], [0.25, 0.5, 0.75])
    log.info(f"Representativeness quartiles frame {q} vs sample {qs}")
else:
    log.info("Representativeness raster not available; skipped")

## 5. Practicality: owner, state, access class, disturbance flags, crew days

In [ ]:
prac = prim.groupby(["level", "access_class"]).size().unstack(fill_value=0)
own = prim.groupby(["level", "owner"]).size().unstack(fill_value=0)
st = prim.groupby(["level", "state"]).size().unstack(fill_value=0)
dist = prim.groupby("level")[["post_fire", "treatment_2027_2031", "remeasure_flag"]].sum()
days_per_class = {1: 0.5, 2: 0.75, 3: 1.0, 4: 2.0}       # crew-days per plot, placeholder until unit costs arrive
crew_days = (prac * pd.Series(days_per_class)).sum(axis=1)
log.info(f"Crew-days by level (cumulative): {crew_days.cumsum().round(1).to_dict()}")
pd.concat({"access": prac, "owner": own, "state": st, "flags": dist}, axis=1).to_csv(O / "eval_practicality.csv")
pd.concat({"access": prac, "state": st, "flags": dist}, axis=1)

## 6. QA report

In [ ]:
checks = {
    "nulls": qa.check_nulls(prim, ["plot_id", "cell_id", "x", "y", "forest_type", "access_class"]),
    "duplicate_plot_ids": qa.check_duplicates(sel, ["plot_id"]),
    "duplicate_units": qa.check_duplicates(sel, ["unit_id"]),
    "sites_closer_than_min_distance": qa.check_min_distance(prim, cfg["draw"]["min_distance_m"]),
    "row_count_primary": qa.check_row_count(prim, expected_min=cfg["allocation"]["levels"]["min"], expected_max=cfg["allocation"]["levels"]["full"] + 50),
    "forest_type_domain": qa.check_value_domain(prim, "forest_type", list(cfg["forest_types"]["population_acres"])),
    "cells_below_floor_min": {"count": int(cov["below_floor_min"].sum())},
    "empty_threshold_classes": {"count": missing},
    "covariate_flags": {"count": len(flag)},
}
for k, v in checks.items(): log.info(f"QA {k}: {v}")
pd.json_normalize(checks).T.to_csv(O / "qa_report.csv", header=False)
log.info("Evaluation complete; review outputs/ before freeze")